# DocLens — Colab Quickstart Notebook

**Fine-tuned Vision-Language Model (Qwen2-VL-2B + QLoRA) for Document Extraction & Tamper Detection**

Repository: [Moinuddinshaik-code/Doclens](https://github.com/Moinuddinshaik-code/Doclens)

This notebook runs the complete end-to-end DocLens pipeline on a free Google Colab T4 GPU:
1. **Environment Setup** — Clone repository and install requirements
2. **Synthetic Data Generation** — Render synthetic receipts and build 7 adversarial test subsets
3. **QLoRA Fine-Tuning** — Fine-tune Qwen2-VL-2B on 4-bit quantized base weights in ~10-15 minutes
4. **Field Extraction & Tamper Detection** — Extract structured JSON and flag forged documents
5. **Benchmark Evaluation** — Measure exact match accuracy and clean vs. adversarial performance delta

## Step 1: Check GPU & Setup Environment

In [ ]:
# Verify GPU availability (T4, V100, or A100)
!nvidia-smi

# Clone or pull latest repository from Moinuddinshaik-code/Doclens
import os
if not os.path.exists('/content/Doclens'):
    !git clone https://github.com/Moinuddinshaik-code/Doclens.git
    %cd /content/Doclens
else:
    %cd /content/Doclens
    !git pull

# Install project dependencies directly
!pip install -q -r requirements.txt

## Step 2: Generate Synthetic Data & Build Adversarial Test Harness

In [ ]:
# Generate 600 train + 100 val + 100 clean test receipt images with ground truth
!python data/synthetic_generator.py

# Build 7 adversarial test subsets (blur, rotation, occlusion, lighting, compression, heavy, tamper)
!python data/eval_harness.py

## Step 3: Run QLoRA Fine-Tuning (Qwen2-VL-2B + 4-bit Quantization)

In [ ]:
# Fine-tune LoRA adapters on Qwen2-VL-2B (takes ~10-15 mins on Colab T4 GPU)
!python training/train.py --config training/config.yaml

## Step 4: Run Field Extraction & Tamper Detection Demo

In [ ]:
from PIL import Image
import json
from doclens.model import load_model
from doclens.extractor import DocLensExtractor

# Display sample receipt image
sample_image = 'data/generated/clean_test/receipt_0700.png'
display(Image.open(sample_image).resize((400, 600)))

# Load fine-tuned model
model, processor = load_model(
    quantization='4bit',
    lora_adapter_path='training/checkpoints/lora_adapter'
)

extractor = DocLensExtractor(model, processor)

# Run extraction & tamper detection
result = extractor.extract_with_tamper(sample_image, document_type='receipt')
print('\nExtracted Structured Output:')
print(json.dumps(result, indent=2))

## Step 5: Full Benchmark Report (Clean vs. Adversarial Performance)

In [ ]:
from doclens.evaluator import DocLensEvaluator

evaluator = DocLensEvaluator(extractor)

# Evaluate across clean test set and all 7 adversarial test subsets
report = evaluator.full_benchmark_report(
    clean_test_dir='data/generated/clean_test',
    adversarial_test_dir='data/generated/adversarial_test',
    output_path='benchmark_report.json'
)